# Functions for data statistics

In [1]:
import pandas as pd
import numpy as np

In [6]:
dummy_context_data = pd.read_csv("../inputs/dummy_context_data.csv")

timestamp_columns = ["observed_at", "time"]
timestamp_column = timestamp_columns[1]
dummy_context_data[timestamp_column] = pd.to_datetime(dummy_context_data[timestamp_column])
timestamps = dummy_context_data[timestamp_column].drop_duplicates().sort_values()

print(f"Min: {timestamps.min()}")
print(f"Max: {timestamps.max()}")
print(f"Unique: {len(timestamps)}")

dummy_context_data["value"] = dummy_context_data["value"].astype(float)
print(dummy_context_data.groupby("attr_name")["value"].agg(["count","min", "max", "mean"]))

Min: 2026-02-11 08:33:00.152000+00:00
Max: 2026-02-12 01:12:00.152000+00:00
Unique: 1000
                      count    min    max      mean
attr_name                                          
leafRelativeHumidity    500   0.00   0.98   0.63156
leafTemperature         500  12.95  45.99  19.70138


In [14]:
def data_preprocessing(dataframe: pd.DataFrame) -> pd.DataFrame:
    pivoted_df = dataframe.pivot_table(
        index=["entity_id", "observed_at","time"],
        columns="attr_name",
        values="value",
        aggfunc="first"
    ).reset_index()
    pivoted_df.columns.name = None
    return pivoted_df.drop(columns="entity_id")
df = data_preprocessing(dummy_context_data)
print(df.head())

                observed_at                             time  \
0  2026-02-11T08:33:00.151Z 2026-02-11 08:33:00.152000+00:00   
1  2026-02-11T08:36:00.151Z 2026-02-11 08:36:00.152000+00:00   
2  2026-02-11T08:38:00.151Z 2026-02-11 08:38:00.152000+00:00   
3  2026-02-11T08:39:00.151Z 2026-02-11 08:39:00.152000+00:00   
4  2026-02-11T08:40:00.151Z 2026-02-11 08:40:00.152000+00:00   

   leafRelativeHumidity  leafTemperature  
0                   NaN            13.49  
1                  0.74            13.42  
2                  0.74            13.67  
3                  0.73              NaN  
4                   NaN            13.50  


In [15]:
df_stats = df.describe()
df_stats

,leafRelativeHumidity,leafTemperature
count,500.000000,500.000000
mean,0.631560,19.701380
std,0.255924,12.586168
min,0.000000,12.950000
25%,0.730000,13.400000
50%,0.740000,13.550000
75%,0.742500,13.792500
max,0.980000,45.990000


In [16]:
from pandas.api.types import is_numeric_dtype
from scipy import stats


def get_stats(df: pd.DataFrame):
    for column in df.columns:
        
            print(f"Statistics for column: {column}")
            #print(df[column].describe())
            print("\n")
            print(f"Count for column {column}: {df[column].count()}")
            print(f"Missing values for column {column}: {df[column].isnull().sum()}")
            print(f"Mean for column {column}: {df[column].mean() if is_numeric_dtype(df[column]) else 'N/A'}")
            print(f"Min for column {column}: {df[column].min() if is_numeric_dtype(df[column]) else 'N/A'}")
            print(f"Max for column {column}: {df[column].max() if is_numeric_dtype(df[column]) else 'N/A'}")
            print(f"Dtype for column {column}: {df[column].dtype}")
            if is_numeric_dtype(df[column]):
                z_scores = stats.zscore(df[column])
                extreme_count = (np.abs(z_scores) > 3).sum()
                print(f"Number of extreme outliers (|z| > 3) for column {column}: {extreme_count}")
            print("\n")

get_stats(df)

Statistics for column: observed_at


Count for column observed_at: 771
Missing values for column observed_at: 0
Mean for column observed_at: N/A
Min for column observed_at: N/A
Max for column observed_at: N/A
Dtype for column observed_at: str


Statistics for column: time


Count for column time: 771
Missing values for column time: 0
Mean for column time: N/A
Min for column time: N/A
Max for column time: N/A
Dtype for column time: datetime64[us, UTC]


Statistics for column: leafRelativeHumidity


Count for column leafRelativeHumidity: 500
Missing values for column leafRelativeHumidity: 271
Mean for column leafRelativeHumidity: 0.6315599999999999
Min for column leafRelativeHumidity: 0.0
Max for column leafRelativeHumidity: 0.98
Dtype for column leafRelativeHumidity: float64
Number of extreme outliers (|z| > 3) for column leafRelativeHumidity: 0


Statistics for column: leafTemperature


Count for column leafTemperature: 500
Missing values for column leafTemperature: 271
Mean for column

In [18]:
def calculate_quality(dataframe: pd.DataFrame, time_col1: str, time_col2: str) -> dict:
    column_names = dataframe.columns.to_list()
    cols_to_remove = [time_col1, time_col2]
    feature_names = [c for c in column_names if c not in cols_to_remove]
    missing_values = dataframe[feature_names].isna().sum().sum()
    completeness = 1 - (missing_values / (len(dataframe) * len(feature_names)))
    dataframe[time_col1] = pd.to_datetime(dataframe[time_col1])
    dataframe[time_col2] = pd.to_datetime(dataframe[time_col2])
    return {
        "number_of_attributes": len(feature_names),
        "missing": missing_values,
        "completeness": completeness,
        "avg_latency": np.abs(dataframe[time_col1] - dataframe[time_col2]).mean().total_seconds(),
    }
calculate_quality(df, time_col1="time", time_col2="observed_at")

{'number_of_attributes': 2,
 'missing': np.int64(542),
 'completeness': np.float64(0.648508430609598),
 'avg_latency': 0.001}

In [19]:
import pandas as pd

# Sample data with roughly 5-minute intervals
df = pd.DataFrame({
    'timestamp': pd.to_datetime(['2026-01-01 10:00:00', '2026-01-01 10:05:02', '2026-01-01 10:10:01'])
})

# 1. Infer frequency
freq = pd.infer_freq(df['timestamp'])
print(f"Inferred Frequency: {freq}")

Inferred Frequency: None


In [5]:
df = df.sort_values('timestamp')

# 2. Calculate the difference between consecutive rows
deltas = df['timestamp'].diff().dropna()

# 3. Find the most common interval (the Mode)
detected_freq = deltas.mode()[0]

clean_freq = pd.to_timedelta(np.round(detected_freq.total_seconds() / 60) * 60, unit='s')

print(f"Detected common interval: {clean_freq}")

# Convert to a string format like '5min'
freq_str = f"{int(clean_freq.total_seconds() // 60)}min"

print(f"Resampling using: {freq_str}")

Detected common interval: 0 days 00:05:00
Resampling using: 5min
